In [1]:
import os
os.chdir("..")
print(os.getcwd())

/work/alicia/NemoLumi


In [2]:
DATASET = "nemotron"
LANG = "eng_Latn"
ID_FIELD = "WARC-Record-ID" # warc_record_id
BENCHMARKS = "/work/alicia/NemoLumi/benchmarks/english_benchmarks.yaml"
NGRAMS_PKL = f"/work/alicia/decontamination_2026/{DATASET}/{LANG}/matched_ngrams/all_matched_ngrams.pkl"
DELETED = f"/work/alicia/decontamination_2026/{DATASET}/{LANG}/analysis/combined_removed_{DATASET}.jsonl"
OUTPUT = "/work/alicia/decontamination_2026/{DATASET}/{LANG}/analysis/removed_by_benchmark.json"
THRESHOLD = 10
print(f"Loading ngrams from '{NGRAMS_PKL}'")
print(f"Loading dataset from '{DATASET}' with lang {LANG}")
print(f"Using benchmarks from '{BENCHMARKS}' and threshold '{THRESHOLD}'")

Loading ngrams from '/work/alicia/decontamination_2026/nemotron/eng_Latn/matched_ngrams/all_matched_ngrams.pkl'
Loading dataset from 'nemotron' with lang eng_Latn
Using benchmarks from '/work/alicia/NemoLumi/benchmarks/english_benchmarks.yaml' and threshold '10'


In [3]:
import pickle
with open(NGRAMS_PKL, "rb") as f:
    x = pickle.load(f)

matched_ngrams = x["matched-ngrams"]
filtered_ngrams = []
total_times = 0
for key, value in x["matched-ngrams"].items():
    if value <= THRESHOLD:
        filtered_ngrams.append(key)
        total_times += value
        
print(len(matched_ngrams))
print(len(filtered_ngrams))
print(total_times)

239465
200252
526695


In [4]:

import yaml
import yaml
import importlib
import sys
import os



def load_class(path: str):
    """
    Given 'package.module.ClassName', return the class object.
    """
    module_path, class_name = path.rsplit('.', 1)
    module = importlib.import_module(module_path)
    return getattr(module, class_name)


def load_yaml_objects(yaml_path: str):
    benchmarks = {}
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    data = data["tasks"]
    objects = []
    for item in data:
        print(item)
        short_name = item["name"].split(".")[-1]
        cls = load_class(item["name"])
        params = item.get("params", {})
        obj = cls(**params)
        objects.append(obj)
        generated_ngrams = obj.generate_ngrams()
        filtered_keys = {x for x in filtered_ngrams}

        generated_ngrams = [
            (x, y) for x, y in generated_ngrams.items()
            if x in filtered_keys
        ]
        #generated_ngrams = [(x, y) for x, y in generated_ngrams.items() if x in filtered_ngrams]
        print(len(generated_ngrams))
        if short_name not in benchmarks:
            benchmarks[short_name] = generated_ngrams
        else:
            benchmarks[short_name].extend(generated_ngrams)
    return benchmarks
benchmarks = load_yaml_objects(BENCHMARKS)

{'name': 'benchmarks.code.english_benchmarks.MMLU', 'params': {'split_type': 'test'}}


/home/alicia/.local/share/virtualenvs/alicia-8Nw4BfrI/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


12829
{'name': 'benchmarks.code.english_benchmarks.COPA', 'params': {'split_type': 'validation'}}
0
{'name': 'benchmarks.code.english_benchmarks.Lambada', 'params': {'split_type': 'test'}}
3428
{'name': 'benchmarks.code.english_benchmarks.OpenBookQA', 'params': {'split_type': 'test'}}
1
{'name': 'benchmarks.code.english_benchmarks.ArcChallenge', 'params': {'split_type': 'test'}}
596
{'name': 'benchmarks.code.english_benchmarks.ArcEasy', 'params': {'split_type': 'test'}}
851
{'name': 'benchmarks.code.english_benchmarks.BoolQ', 'params': {'split_type': 'validation'}}
51300
{'name': 'benchmarks.code.english_benchmarks.HellaSwag', 'params': {'split_type': 'validation'}}
11168
{'name': 'benchmarks.code.english_benchmarks.CommonsenseQA', 'params': {'split_type': 'validation'}}
10
{'name': 'benchmarks.code.english_benchmarks.PIQA', 'params': {'split_type': 'validation'}}
19
{'name': 'benchmarks.code.english_benchmarks.GSM8K', 'params': {'split_type': 'test'}}
49
{'name': 'benchmarks.code.engl

In [7]:
def word_ngrams_from_list(words, min_len=8, max_len=13):
    """
    Generate word n-grams from a pre-tokenized list of words.
    
    Args:
        words (list of str): The list of words from get_words.
        min_len (int): Minimum n-gram length (number of words).
        max_len (int): Maximum n-gram length (number of words).

    Yields:
        str: A word n-gram joined by spaces.
    """
    for n in range(min_len, max_len + 1):
        for i in range(len(words) - n + 1):
            yield " ".join(words[i:i+n])

In [ ]:
from pathlib import Path 
import json
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import os
from nemo_curator.utils.text_utils import get_words

def check_ngrams(text_ngrams):
    text_ng_set = set(text_ngrams)
    
    for benchmark, ngrams in benchmarks.items():
        for ngram_processed, ngram_original in ngrams:
            if ngram_processed in text_ng_set:
                return {
                    "benchmark": benchmark,
                    "matched_ngram": ngram_processed,
                    "benchmark_text": ngram_original
                }
    return "Not Found"

BENCHMARKS = "/work/alicia/NemoLumi/benchmarks/english_benchmarks.yaml"
NGRAMS_PKL = f"/work/alicia/decontamination_2026/{DATASET}/{LANG}/matched_ngrams/all_matched_ngrams.pkl"
DELETED_DIR = f"/work/alicia/decontamination_2026/{DATASET}/{LANG}/final_removed_data"
OUTPUT = f"/work/alicia/decontamination_2026/{DATASET}/{LANG}/analysis/removed_by_benchmark.jsonl"
FILE_OUTPUTS = f"/work/alicia/decontamination_2026/{DATASET}/{LANG}/analysis/combined"
file_dir=DELETED_DIR
file_outputs = FILE_OUTPUTS
Path(file_outputs).mkdir(parents=True, exist_ok=True)
final_output_results = OUTPUT
def process_file(file):
    not_detected_lines = []
    output_filepath = Path(file_outputs) / Path(file).parent.name
    output_filepath.mkdir(parents=True, exist_ok=True)
    output_file = output_filepath / f"{file.stem}_output.jsonl"
    Path(output_file).unlink(missing_ok=True)
    with open(file, "r", encoding="utf-8") as f, open(output_file, "w") as out_f:
        for i,line in enumerate(f):
            myjson = json.loads(line)
            text = myjson["text"]
            text_ngrams, _ = get_words(text)
            text_cleaned = word_ngrams_from_list(text_ngrams)
            outputs = check_ngrams(text_cleaned)
            if outputs == "Not Found":
                print(f"NOT DETECTED {i, file}")
                not_detected_lines.append(f"{i}, {str(file)}")
            else:
                final_output = {
                        "warc_record_id": myjson["metadata"][ID_FIELD],
                        "file_part":  file.stem.split,
                        "benchmark": outputs["benchmark"], "matched_ngram": outputs["matched_ngram"], "benchmark_text": outputs["benchmark_text"],
                        "train": text,
                    }
                out_f.write(json.dumps(final_output)+"\n")
    return not_detected_lines


def main():
    file_paths = list(Path(file_dir).rglob("*.jsonl"))
    with ProcessPoolExecutor() as executor:
        futures = {executor.submit(process_file, file): file for file in file_paths}
        with open("test_output.txt", "a") as output_file:
            for future in as_completed(futures):
                for line in future.result():
                    output_file.write(line+"\n")
        
    with open(final_output_results, "w") as outfile:
        search_here = list(Path(file_outputs).rglob("*.jsonl"))
        for fname in search_here:
            if str(fname).endswith(".jsonl"):
                fpath = os.path.join(file_outputs, fname)
                
                with open(fpath, "r", encoding="utf-8") as infile:
                    for line in infile:
                        try:
                            obj = json.loads(line)
                            clean = json.dumps(obj, ensure_ascii=False)
                            outfile.write(clean + "\n")
                        except json.JSONDecodeError:
                            print(f"Skipping invalid JSON line in {fname}")
main()    

In [4]:
import polars as pl
final_output_results = f"/work/alicia/decontamination_2026/{DATASET}/{LANG}/analysis/removed_by_benchmark.jsonl"
pl.Config.set_fmt_str_lengths(1000)
df = pl.read_ndjson(final_output_results)
filtered_df = df.filter(pl.col("benchmark").is_in(["Belebele"]))

In [5]:
print(filtered_df)

shape: (9_313, 6)
┌─────────────────┬─────────────────┬───────────┬────────────────┬────────────────┬────────────────┐
│ warc_record_id  ┆ file_part       ┆ benchmark ┆ matched_ngram  ┆ benchmark_text ┆ train          │
│ ---             ┆ ---             ┆ ---       ┆ ---            ┆ ---            ┆ ---            │
│ str             ┆ str             ┆ str       ┆ str            ┆ str            ┆ str            │
╞═════════════════╪═════════════════╪═══════════╪════════════════╪════════════════╪════════════════╡
│ 6d7192f2-c828-4 ┆ CC-MAIN-2014-42 ┆ Belebele  ┆ the results of ┆ "Former House  ┆ Former         │
│ be8-ab08-f7f950 ┆ -part-00012     ┆           ┆ tonights       ┆ Speaker Newt   ┆ Massachusetts  │
│ b9c461          ┆                 ┆           ┆ caucus         ┆ Gingrich,      ┆ governor Mitt  │
│                 ┆                 ┆           ┆ determine      ┆ Texas governor ┆ Romney has won │
│                 ┆                 ┆           ┆ whether there  ┆ Rick P

In [ ]:
def check_ngrams_all(text):
    all_hits = []
    for benchmark, values in benchmarks.items():
        ngrams = values
        for ngram_proc, ngram_orig in ngrams:
            if ngram_proc in text:
                all_hits.append({"benchmark": benchmark, "matched_ngram": ngram_proc, "benchmark_text": ngram_orig})
        
    return all_hits


from pathlib import Path 
import json

from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import os
from nemo_curator.utils.text_utils import get_words
file_dir="nemotron_100_removedv2"
file_outputs = "analysis_results/all_hits"
Path(file_outputs).mkdir(parents=True, exist_ok=True)
final_output_results = "analysis_results/nemotron_sample/all_collisions.jsonl"
def process_file(file):
    not_detected_lines = []
    output_file = Path(file_outputs) / f"{file.stem}_output.jsonl"
    Path(output_file).unlink(missing_ok=True)
    with open(file, "r", encoding="utf-8") as f, open(output_file, "w") as out_f:
        for i,line in enumerate(f):
            myjson = json.loads(line)
            text = myjson["text"]
            text_ngrams, _ = get_words(text)
            text_cleaned = " ".join(text_ngrams)
            outputs = check_ngrams_all(text_cleaned)
            if outputs == "Not Found":
                print(f"NOT DETECTED {i, file}")
                not_detected_lines.append(f"{i}, {str(file)}")
            else:
                for output in outputs:
                    final_output = {
                        "warc_record_id": myjson["warc_record_id"],
                        "file_part": file.stem.split("_")[0],
                        "benchmark": output["benchmark"], "matched_ngram": output["matched_ngram"], "benchmark_text": output["benchmark_text"],
                        "train": text,
                    }
                    out_f.write(json.dumps(final_output)+"\n")
    return not_detected_lines


def main():
    file_paths = list(Path(file_dir).glob("*.jsonl"))
    with ProcessPoolExecutor() as executor:
        futures = {executor.submit(process_file, file): file for file in file_paths}
        with open("test_output.txt", "a") as output_file:
            for future in as_completed(futures):
                for line in future.result():
                    output_file.write(line+"\n")
        
    with open(final_output_results, "w") as outfile:
        # Iterate through files in directory
        for fname in os.listdir(file_outputs):
            if fname.endswith(".jsonl"):
                fpath = os.path.join(file_outputs, fname)
                
                with open(fpath, "r", encoding="utf-8") as infile:
                    for line in infile:
                        try:
                            obj = json.loads(line)
                            clean = json.dumps(obj, ensure_ascii=False)
                            outfile.write(clean + "\n")
                        except json.JSONDecodeError:
                            print(f"Skipping invalid JSON line in {fname}")
                            
main()